In [1]:
import pandas as pd

In [2]:
INPUT = "RDV_MMS_all_overlaps.csv"
BINS = {"10kb", "50kb", "500kb", "1000kb"}

In [3]:
df = pd.read_csv(INPUT)

In [4]:
#explode gene list column
df["gene_name"] = (
    df["gene_name"].astype(str)
    .str.strip("[]").str.replace("'", "", regex=False).str.replace(",", "", regex=False)
    .str.replace(",", "", regex=False).str.split()
)
df = df.explode("gene_name")

In [5]:
# extract patient + bin
df["patient"] = df["sample"].str.extract(r"^(RDV|MM|MMS)")[0].replace({"MM": "MMS"})
df["bin"] = df["sample"].str.extract(r"(10kb|50kb|500kb|1000kb)")[0]

In [6]:
#group by patient, chr, gene_name
grp = (
    df.groupby(["patient", "chr", "gene_name"])
    .agg(
        bins=("bin", lambda x: set(x)),
        events=("event", lambda x: sorted(set(x)))
    )
    .reset_index()
)

In [7]:
#only genes present in all 4 bins
grp = grp[grp["bins"].apply(lambda b: b >= BINS)]

In [8]:
#expand events into comma-separated string
grp["events"] = grp["events"].apply(lambda ev: ",".join(ev))

In [9]:
#drop bins column
grp = grp[["patient", "chr", "gene_name", "events"]]

In [10]:
#patient detailed tables
for patient in grp["patient"].unique():
    out = grp[grp["patient"] == patient][["chr", "gene_name", "events"]]
    out.to_csv(f"{patient}_consistent_chr_gene_all4bins.tsv", sep="\t", index=False)

In [13]:
#explode events again since some genes may have multiple events
event_expanded = grp.assign(event=grp["events"].str.split(",")).explode("event")

In [14]:
summary = (
    event_expanded.groupby(["patient", "event"])
    .size()
    .reset_index(name="count")
    .pivot(index="patient", columns="event", values="count")
    .fillna(0)
    .astype(int)
)

In [15]:
summary.to_csv("consistent_gene_event_counts.tsv", sep="\t")

In [16]:
summary

event,AMP,GAIN,HETD,HLAMP,HLAMP2,HLAMP3
patient,,,,,,
MMS,36,8,78,6,23,35
RDV,42,52,123,0,10,0


In [ ]:
#count events per chromosome per patient
chr_event_counts = (
    event_expanded.groupby(["patient", "chr", "event"])
    .size()
    .reset_index(name="count")
    .sort_values(["patient", "chr", "event"])
)

In [ ]:
chr_event_counts.to_csv("consistent_chr_event_counts.tsv", sep="\t", index=False)

In [ ]:
#patient wide tables
for patient in chr_event_counts["patient"].unique():
    sub = chr_event_counts[chr_event_counts["patient"] == patient]
    wide = (
        sub.pivot_table(index="chr", columns="event", values="count", aggfunc="sum", fill_value=0)
    )
    wide.to_csv(f"{patient}_chr_event_counts.tsv", sep="\t")
    print(patient)
    print(wide)

MMS
event  AMP  GAIN  HETD  HLAMP  HLAMP2  HLAMP3
chr                                          
chr1     0     2    11      0       0       0
chr19    0     6    30      0       0       0
chr21    5     0     0      0       4       5
chr4     0     0     1      0       0       0
chr5    31     0     0      6      19      30
chrX     0     0    36      0       0       0
RDV
event  AMP  GAIN  HETD  HLAMP2
chr                           
chr10    0     0     2       0
chr13    0     0    14       0
chr14    7     1     0       1
chr18    0     0     0       9
chr19   21     2    10       0
chr2     0    48     1       0
chr3     0     0    51       0
chr9    14     1     9       0
chrX     0     0    36       0
